In [0]:
# Governança de Dados

#Este notebook implementa controles de governança para o projeto Logistics Operations Intelligence, abrangendo catálogo de dados, qualidade, linhagem e rastreabilidade das transformações entre as camadas Bronze, Silver e Gold.

##Inventário automático das tabelas

##Inventariar tabelas das camadas do projeto

In [0]:
from pyspark.sql import functions as F

In [0]:
schemas_projeto = [
    ("bronze", "workspace.bronze_logistics"),
    ("silver", "workspace.silver_logistics"),
    ("gold", "workspace.gold_logistics")
]

inventario = []

for camada, schema in schemas_projeto:
    
    tabelas = spark.sql(
        f"SHOW TABLES IN {schema}"
    ).collect()
    
    for tabela in tabelas:
        inventario.append(
            (
                camada,
                schema,
                tabela.tableName,
                f"{schema}.{tabela.tableName}"
            )
        )

##Criar DataFrame do catálogo técnico

In [0]:
df_catalogo_tecnico = spark.createDataFrame(
    inventario,
    [
        "layer",
        "schema_name",
        "table_name",
        "full_table_name"
    ]
)

display(
    df_catalogo_tecnico.orderBy(
        "layer",
        "table_name"
    )
)

##Adicionar metadados de governança

##Definir metadados das principais tabelas

In [0]:
catalogo_negocio = [
    
    (
        "trips",
        "silver",
        "Viagens realizadas pela operação logística.",
        "Uma linha por viagem.",
        "trip_id",
        "load_id, driver_id, truck_id, trailer_id",
        "Operações Logísticas"
    ),
    
    (
        "loads",
        "silver",
        "Cargas transportadas pela operação.",
        "Uma linha por carga.",
        "load_id",
        "customer_id, route_id",
        "Operações Logísticas"
    ),
    
    (
        "routes",
        "silver",
        "Cadastro e características das rotas logísticas.",
        "Uma linha por rota.",
        "route_id",
        "",
        "Planejamento Logístico"
    ),
    
    (
        "drivers",
        "silver",
        "Cadastro e informações profissionais dos motoristas.",
        "Uma linha por motorista.",
        "driver_id",
        "",
        "Gestão de Motoristas"
    ),
    
    (
        "trucks",
        "silver",
        "Cadastro dos veículos pertencentes à frota.",
        "Uma linha por caminhão.",
        "truck_id",
        "",
        "Gestão de Frota"
    ),
    
    (
        "delivery_events",
        "silver",
        "Eventos programados e realizados de coleta e entrega.",
        "Uma linha por evento logístico.",
        "event_id",
        "load_id, trip_id, facility_id",
        "Operações Logísticas"
    ),
    
    (
        "fuel_purchases",
        "silver",
        "Compras de combustível associadas às viagens.",
        "Uma linha por abastecimento.",
        "fuel_purchase_id",
        "trip_id, truck_id, driver_id",
        "Gestão de Combustível"
    ),
    
    (
        "gold_trip_operations",
        "gold",
        "Visão analítica consolidada das operações de viagem.",
        "Uma linha por viagem.",
        "trip_id",
        "load_id, route_id, driver_id, truck_id",
        "Analytics"
    ),
    
    (
        "gold_delivery_performance",
        "gold",
        "Indicadores de desempenho de coleta e entrega.",
        "Uma linha por viagem.",
        "trip_id",
        "",
        "Analytics"
    ),
    
    (
        "gold_fuel_performance",
        "gold",
        "Indicadores de consumo e custo de combustível.",
        "Uma linha por viagem.",
        "trip_id",
        "",
        "Analytics"
    )
]

##Catálogo de dados

In [0]:
df_catalogo_dados = spark.createDataFrame(
    catalogo_negocio,
    [
        "table_name",
        "layer",
        "business_description",
        "grain",
        "primary_key",
        "foreign_keys",
        "data_domain"
    ]
)

display(df_catalogo_dados)

##Gravar catálogo de dados

In [0]:
(
    df_catalogo_dados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.governance_logistics.data_catalog"
    )
)

##Catálogo de regras de qualidade

##Definir regras de qualidade

In [0]:
regras_qualidade = [
    
    (
        "DQ001",
        "silver.trips",
        "Unicidade",
        "trip_id deve ser único.",
        "Crítica"
    ),
    
    (
        "DQ002",
        "silver.trips",
        "Completude",
        "trip_id não pode ser nulo.",
        "Crítica"
    ),
    
    (
        "DQ003",
        "silver.loads",
        "Integridade",
        "customer_id deve existir em customers.",
        "Crítica"
    ),
    
    (
        "DQ004",
        "silver.loads",
        "Integridade",
        "route_id deve existir em routes.",
        "Crítica"
    ),
    
    (
        "DQ005",
        "silver.delivery_events",
        "Integridade",
        "trip_id deve existir em trips.",
        "Crítica"
    ),
    
    (
        "DQ006",
        "silver.delivery_events",
        "Consistência",
        "Cada viagem deve possuir um evento Pickup e um evento Delivery.",
        "Crítica"
    ),
    
    (
        "DQ007",
        "silver.delivery_events",
        "Consistência",
        "on_time_flag deve ser comparado com o indicador recalculado.",
        "Alerta"
    ),
    
    (
        "DQ008",
        "silver.fuel_purchases",
        "Validade",
        "total_cost deve ser consistente com gallons multiplicado por price_per_gallon.",
        "Crítica"
    ),
    
    (
        "DQ009",
        "silver.fuel_purchases",
        "Completude",
        "Monitorar compras sem truck_id.",
        "Alerta"
    ),
    
    (
        "DQ010",
        "silver.fuel_purchases",
        "Completude",
        "Monitorar compras sem driver_id.",
        "Alerta"
    ),
    
    (
        "DQ011",
        "gold.gold_trip_operations",
        "Unicidade",
        "trip_id deve permanecer único na camada Gold.",
        "Crítica"
    ),
    
    (
        "DQ012",
        "gold.gold_delivery_performance",
        "Unicidade",
        "Cada viagem deve possuir apenas um registro analítico.",
        "Crítica"
    ),
    
    (
        "DQ013",
        "gold.gold_fuel_performance",
        "Unicidade",
        "Cada viagem deve possuir apenas um registro analítico.",
        "Crítica"
    )
]

##Criar DataFrame das regras

In [0]:
df_regras_qualidade = spark.createDataFrame(
    regras_qualidade,
    [
        "rule_id",
        "table_name",
        "quality_dimension",
        "rule_description",
        "severity"
    ]
)

display(df_regras_qualidade)

##Gravar catálogo de regras

In [0]:
(
    df_regras_qualidade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.governance_logistics.data_quality_rules"
    )
)

##Registrar os achados reais de qualidade

In [0]:
resultados_qualidade = [
    
    (
        "DQ001",
        "silver.trips",
        "PASS",
        0,
        "Nenhuma duplicidade de trip_id encontrada."
    ),
    
    (
        "DQ002",
        "silver.trips",
        "PASS",
        0,
        "Nenhum trip_id nulo."
    ),
    
    (
        "DQ006",
        "silver.delivery_events",
        "PASS",
        0,
        "Todas as viagens possuem Pickup e Delivery."
    ),
    
    (
        "DQ007",
        "silver.delivery_events",
        "WARNING",
        56866,
        "Divergências encontradas entre on_time_flag da fonte e indicador recalculado."
    ),
    
    (
        "DQ008",
        "silver.fuel_purchases",
        "PASS",
        0,
        "Todos os custos foram considerados consistentes dentro da tolerância definida."
    ),
    
    (
        "DQ009",
        "silver.fuel_purchases",
        "WARNING",
        3880,
        "Compras de combustível sem truck_id."
    ),
    
    (
        "DQ010",
        "silver.fuel_purchases",
        "WARNING",
        3988,
        "Compras de combustível sem driver_id."
    ),
    
    (
        "DQ011",
        "gold.gold_trip_operations",
        "PASS",
        0,
        "Granularidade de uma linha por viagem preservada."
    ),
    
    (
        "DQ012",
        "gold.gold_delivery_performance",
        "PASS",
        0,
        "Granularidade de uma linha por viagem preservada."
    ),
    
    (
        "DQ013",
        "gold.gold_fuel_performance",
        "PASS",
        0,
        "Granularidade de uma linha por viagem preservada."
    )
]

##Criar DataFrame dos resultados

In [0]:
df_resultados_qualidade = spark.createDataFrame(
    resultados_qualidade,
    [
        "rule_id",
        "table_name",
        "status",
        "affected_records",
        "observation"
    ]
)

display(df_resultados_qualidade)

##Gravar resultados de qualidade

In [0]:
(
    df_resultados_qualidade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.governance_logistics.data_quality_results"
    )
)

##Criar linhagem das tabelas Gold

##Definir linhagem de dados

In [0]:
linhagem_dados = [
    
    (
        "workspace.gold_logistics.gold_trip_operations",
        "workspace.silver_logistics.trips",
        "Join e enriquecimento"
    ),
    
    (
        "workspace.gold_logistics.gold_trip_operations",
        "workspace.silver_logistics.loads",
        "Join e enriquecimento"
    ),
    
    (
        "workspace.gold_logistics.gold_trip_operations",
        "workspace.silver_logistics.routes",
        "Join e enriquecimento"
    ),
    
    (
        "workspace.gold_logistics.gold_trip_operations",
        "workspace.silver_logistics.drivers",
        "Join e enriquecimento"
    ),
    
    (
        "workspace.gold_logistics.gold_trip_operations",
        "workspace.silver_logistics.trucks",
        "Join e enriquecimento"
    ),
    
    (
        "workspace.gold_logistics.gold_delivery_performance",
        "workspace.silver_logistics.delivery_events",
        "Agregação e pivot lógico de Pickup e Delivery"
    ),
    
    (
        "workspace.gold_logistics.gold_fuel_performance",
        "workspace.silver_logistics.fuel_purchases",
        "Agregação por viagem"
    ),
    
    (
        "workspace.gold_logistics.gold_fuel_performance",
        "workspace.silver_logistics.trips",
        "Enriquecimento operacional"
    )
]

##Criar DataFrame da linhagem

In [0]:
df_linhagem = spark.createDataFrame(
    linhagem_dados,
    [
        "target_table",
        "source_table",
        "transformation_type"
    ]
)

display(df_linhagem)

##Gravar tabela de linhagem

In [0]:
(
    df_linhagem.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.governance_logistics.data_lineage"
    )
)

##Dicionário de dados

In [0]:
dicionario_dados = [

    (
        "silver.trips",
        "trip_id",
        "string",
        "Identificador único da viagem.",
        "Campo proveniente da fonte.",
        "Não",
        "Chave primária da tabela."
    ),

    (
        "silver.trips",
        "calculated_mpg",
        "double",
        "Eficiência de combustível calculada da viagem em milhas por galão.",
        "actual_distance_miles / fuel_gallons_used",
        "Sim",
        "Calculado apenas quando o consumo de combustível é maior que zero."
    ),

    (
        "silver.trips",
        "average_speed_mph",
        "double",
        "Velocidade média calculada da viagem em milhas por hora.",
        "actual_distance_miles / actual_duration_hours",
        "Sim",
        "Calculado quando a duração da viagem é maior que zero."
    ),

    (
        "silver.trips",
        "idle_time_percentage",
        "double",
        "Percentual da duração da viagem correspondente a tempo ocioso.",
        "idle_time_hours / actual_duration_hours * 100",
        "Sim",
        "Definido como nulo quando o tempo ocioso é inconsistente."
    ),

    (
        "silver.delivery_events",
        "event_variance_minutes",
        "double",
        "Diferença em minutos entre o horário realizado e o horário programado.",
        "actual_datetime - scheduled_datetime",
        "Não",
        "Valor positivo representa atraso e valor negativo representa antecipação."
    ),

    (
        "silver.delivery_events",
        "is_on_time_calculated",
        "boolean",
        "Indicador de pontualidade recalculado a partir dos horários.",
        "actual_datetime <= scheduled_datetime",
        "Não",
        "Mantido separadamente do indicador recebido da fonte."
    ),

    (
        "silver.delivery_events",
        "on_time_flag_consistent",
        "boolean",
        "Indica se o indicador de pontualidade da fonte coincide com o indicador recalculado.",
        "on_time_flag = is_on_time_calculated",
        "Não",
        "Utilizado para monitoramento de consistência."
    ),

    (
        "silver.fuel_purchases",
        "calculated_total_cost",
        "double",
        "Custo total recalculado da compra de combustível.",
        "gallons * price_per_gallon",
        "Não",
        "Usado para validar o custo informado pela fonte."
    ),

    (
        "silver.fuel_purchases",
        "has_complete_resource_assignment",
        "boolean",
        "Indica se a compra possui caminhão e motorista identificados.",
        "truck_id IS NOT NULL AND driver_id IS NOT NULL",
        "Não",
        "Indicador de completude."
    ),

    (
        "gold.gold_trip_operations",
        "distance_variance_miles",
        "double",
        "Diferença entre a distância efetivamente percorrida e a distância típica da rota.",
        "actual_distance_miles - typical_distance_miles",
        "Sim",
        "Utilizada para avaliar desvios operacionais."
    ),

    (
        "gold.gold_trip_operations",
        "estimated_operational_margin",
        "double",
        "Margem operacional estimada da viagem.",
        "total_revenue - estimated_route_cost",
        "Sim",
        "É uma estimativa, pois utiliza o custo estimado da rota."
    ),

    (
        "gold.gold_trip_operations",
        "estimated_margin_percentage",
        "double",
        "Percentual estimado de margem sobre a receita da viagem.",
        "(total_revenue - estimated_route_cost) / total_revenue * 100",
        "Sim",
        "Calculado apenas quando a receita é maior que zero."
    ),

    (
        "gold.gold_delivery_performance",
        "total_detention_minutes",
        "double",
        "Tempo total de detenção considerando coleta e entrega.",
        "pickup_detention_minutes + delivery_detention_minutes",
        "Não",
        "Indicador operacional da viagem."
    ),

    (
        "gold.gold_delivery_performance",
        "delivery_cycle_hours",
        "double",
        "Tempo decorrido entre a realização da coleta e a realização da entrega.",
        "delivery_actual_datetime - pickup_actual_datetime",
        "Não",
        "Calculado em horas."
    ),

    (
        "gold.gold_delivery_performance",
        "has_source_flag_inconsistency",
        "boolean",
        "Indica se existe divergência entre o indicador de pontualidade da fonte e o indicador recalculado.",
        "NOT pickup_flag_consistent OR NOT delivery_flag_consistent",
        "Não",
        "Indicador de qualidade herdado da Silver."
    ),

    (
        "gold.gold_fuel_performance",
        "total_gallons_purchased",
        "double",
        "Quantidade total de combustível comprado durante a viagem.",
        "SUM(gallons) por trip_id",
        "Sim",
        "Agregado a partir de fuel_purchases."
    ),

    (
        "gold.gold_fuel_performance",
        "total_fuel_cost",
        "double",
        "Custo total de combustível associado à viagem.",
        "SUM(total_cost) por trip_id",
        "Sim",
        "Agregado a partir das compras de combustível."
    ),

    (
        "gold.gold_fuel_performance",
        "fuel_cost_per_mile",
        "double",
        "Custo de combustível por milha percorrida.",
        "total_fuel_cost / actual_distance_miles",
        "Sim",
        "Calculado quando a distância percorrida é maior que zero."
    )
]

##Criar DataFrame do dicionário

In [0]:
df_dicionario_dados = spark.createDataFrame(
    dicionario_dados,
    [
        "table_name",
        "column_name",
        "data_type",
        "business_description",
        "calculation_rule",
        "nullable",
        "governance_observation"
    ]
)

display(df_dicionario_dados)

##Gravar o dicionário na camada de governança

In [0]:
(
    df_dicionario_dados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.governance_logistics.data_dictionary"
    )
)

##Validar gravação

In [0]:
df_dicionario_gravado = spark.table(
    "workspace.governance_logistics.data_dictionary"
)

print(
    "Total de campos documentados: "
    f"{df_dicionario_gravado.count()}"
)

display(
    df_dicionario_gravado.orderBy(
        "table_name",
        "column_name"
    )
)